# Detectree2: Tree Crown Detection with Adaptive Thresholding
This notebook:
1. Runs Detectree2 inference on all orthomosaics in `input/input_om/`
2. Sweeps confidence thresholds to find the value per OM that yields crown counts closest to the global median
3. Saves the final crowns to `output/input_crowns/OM{n}.gpkg`

In [ ]:
# Optional installs (run only if packages are missing)
import sys, subprocess

def ensure(pkg_args):
    if isinstance(pkg_args, str):
        pkg_args = [pkg_args]
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pkg_args])
    except Exception as e:
        print(f'Failed to install {pkg_args}: {e}')

# Try imports; if missing, attempt installs.
try:
    import rasterio
except Exception:
    ensure(['rasterio'])

try:
    import geopandas as gpd
except Exception:
    ensure(['geopandas', 'pyproj', 'shapely', 'fiona'])

# Detectron2 stack (may already be present in the environment).
try:
    import detectron2
except Exception:
    # Ensure PyTorch CPU build (safe on macOS) before detectron2
    try:
        import torch
    except Exception:
        ensure(['torch', 'torchvision', 'torchaudio'])
    # Try installing detectron2 from source; may take time.
    ensure(['git+https://github.com/facebookresearch/detectron2.git'])

try:
    import detectree2
except Exception:
    ensure(['git+https://github.com/PatBall1/detectree2.git'])

print('Environment check complete.')

In [ ]:
# Diagnostics: confirm kernel + pandas import path
import sys
print("sys.executable:", sys.executable)
print("sys.path[0]:", sys.path[0])
try:
    import pandas as pd
    print("pandas:", pd.__version__, pd.__file__)
except Exception as e:
    print("pandas import failed:", repr(e))

In [ ]:
# Core detection utilities
import os
import warnings

# Thread controls (CPU inference)
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"
import torch
torch.set_num_threads(6)
torch.set_num_interop_threads(6)

import rasterio
import geopandas as gpd

from detectree2.preprocessing.tiling import tile_data
from detectree2.models.train import setup_cfg
from detectree2.models.predict import predict_on_data
from detectree2.models.outputs import project_to_geojson, stitch_crowns, clean_crowns
from detectron2.engine import DefaultPredictor

def tile_orthomosaic(img_path, tiles_path, buffer=20, tile_width=45, tile_height=45, dtype_bool=True):
    os.makedirs(tiles_path, exist_ok=True)
    tile_data(img_path, tiles_path, buffer, tile_width, tile_height, dtype_bool=dtype_bool)

def setup_detection_config(trained_model_path, device='cpu'):
    cfg = setup_cfg(update_model=trained_model_path)
    try:
        cfg.MODEL.DEVICE = device
    except Exception:
        pass
    return cfg

def predict_tree_crowns(tiles_path, cfg):
    predictor = DefaultPredictor(cfg)
    predict_on_data(directory=tiles_path, predictor=predictor)

def reproject_predictions(tiles_path):
    predictions_folder = os.path.join(tiles_path, 'predictions/')
    geo_predictions_folder = os.path.join(tiles_path, 'predictions_geo/')
    project_to_geojson(tiles_path, predictions_folder, geo_predictions_folder)
    return geo_predictions_folder

def tree_crown_detection_pipeline(img_path, tiles_path, trained_model_path,
                                   buffer=20, tile_width=45, tile_height=45, dtype_bool=True,
                                   device='cpu'):
    # Tile
    tile_orthomosaic(img_path, tiles_path, buffer, tile_width, tile_height, dtype_bool)
    # Config
    cfg = setup_detection_config(trained_model_path, device=device)
    # Predict
    predict_tree_crowns(tiles_path, cfg)
    # Reproject
    geo_predictions_folder = reproject_predictions(tiles_path)
    return geo_predictions_folder

In [ ]:
# Step 1: Run detection for all orthomosaics
import os
import glob
import re

project_root = '/Users/hbot07/VS Code/Drone-Phenology-Monitoring'
input_dir = os.path.join(project_root, 'input')
om_dir = os.path.join(input_dir, 'input_om')
tiles_dir = os.path.join(input_dir, 'tiles')
model_path = os.path.join(input_dir, 'detectree_models', '250312_flexi.pth')

# Collect orthomosaics
om_paths = sorted(glob.glob(os.path.join(om_dir, '*.tif')))
if not om_paths:
    raise FileNotFoundError(f"No orthomosaics found in {om_dir}")

print(f"Found {len(om_paths)} orthomosaics. Running detection...")

# Run detection pipeline (inference only)
params = dict(buffer=20, tile_width=45, tile_height=45, dtype_bool=True, device='cpu')

for om_path in om_paths[-2:]:
    stem = os.path.splitext(os.path.basename(om_path))[0]
    tiles_path = os.path.join(tiles_dir, stem)
    geo_predictions_folder = os.path.join(tiles_path, 'predictions_geo')
    
    print(f"\n=== Processing {stem} ===")
    print("Orthomosaic:", om_path)
    print("Tiles dir  :", tiles_path)
    
    # Skip if predictions already exist
    if os.path.isdir(geo_predictions_folder) and glob.glob(os.path.join(geo_predictions_folder, '*.geojson')):
        print(f"Predictions already exist, skipping detection: {geo_predictions_folder}")
        continue
    
    # Run detection (returns path to predictions_geo/)
    geo_predictions_folder = tree_crown_detection_pipeline(
        img_path=om_path,
        tiles_path=tiles_path,
        trained_model_path=model_path,
        **params
    )
    print(f"Predictions saved to: {geo_predictions_folder}")

print("\nDetection complete for all orthomosaics.")

In [ ]:
# Step 2: Confidence sweep and adaptive threshold selection
import numpy as np
import pandas as pd
from detectree2.models.outputs import stitch_crowns, clean_crowns

# Fix PROJ data directory issue
import pyproj
try:
    pyproj_datadir = pyproj.datadir.get_data_dir()
except Exception:
    import subprocess
    result = subprocess.run(['conda', 'info', '--base'], capture_output=True, text=True)
    conda_base = result.stdout.strip()
    proj_data = os.path.join(conda_base, 'share', 'proj')
    if os.path.exists(proj_data):
        pyproj.datadir.set_data_dir(proj_data)
        print(f"Set PROJ_DATA to: {proj_data}")
    else:
        print("Warning: PROJ data directory not found, proceeding anyway...")

# Fixed parameters
fixed_iou = 0.7
fixed_simplify = 0.3
conf_list = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

# Output directory
output_crowns_dir = os.path.join(project_root, 'output', 'input_crowns')
os.makedirs(output_crowns_dir, exist_ok=True)

# Naming function: sit_om{n}.tif -> OM{n}.gpkg
om_num_pat = re.compile(r"sit_om(\d+)", re.IGNORECASE)

def out_name_for(stem: str) -> str:
    m = om_num_pat.search(stem)
    if m:
        return f"OM{int(m.group(1))}.gpkg"
    return f"{stem}_crowns.gpkg"

def _clean_bad_geojson_names(geo_folder: str) -> int:
    bad = glob.glob(os.path.join(geo_folder, '*_None.geojson'))
    for p in bad:
        try:
            os.remove(p)
        except Exception as e:
            print(f"Warning: could not remove {p}: {e}")
    return len(bad)

def load_stitched_for(stem: str):
    geo_folder = os.path.join(tiles_dir, stem, 'predictions_geo')
    if not (os.path.isdir(geo_folder) and glob.glob(os.path.join(geo_folder, '*.geojson'))):
        return None
    removed = _clean_bad_geojson_names(geo_folder)
    if removed:
        print(f"Removed {removed} invalid geojson files in {geo_folder}")
    gdf = stitch_crowns(geo_folder, 1)
    gdf = gdf[gdf.is_valid]
    return gdf

def compute_count_for_conf(base: gpd.GeoDataFrame, conf: float) -> int:
    g = base.copy()
    if fixed_simplify and fixed_simplify > 0:
        g = g.set_geometry(g.geometry.simplify(fixed_simplify))
    g = clean_crowns(g, fixed_iou, conf)
    g = g[g.is_valid]
    return int(len(g))

# Collect orthomosaic stems
stems = [os.path.splitext(os.path.basename(p))[0] for p in om_paths]

print("Running confidence sweep...")
rows = []
bases = {}

for stem in stems:
    base = load_stitched_for(stem)
    if base is None:
        print(f"Skipping {stem}: predictions not found")
        continue
    bases[stem] = base
    for conf in conf_list:
        count = compute_count_for_conf(base, conf)
        rows.append({'orthomosaic': stem, 'confidence': conf, 'count': count})

if not rows:
    raise RuntimeError('No stitched predictions found. Run detection first.')

sweep_df = pd.DataFrame(rows)
target_count = int(np.median(sweep_df['count']))
print(f"Global median count across all OM×conf: {target_count}")

# Select confidence per OM closest to target
def choose_conf_for_counts(df: pd.DataFrame, target: int) -> pd.Series:
    diffs = (df['count'] - target).abs()
    min_diff = diffs.min()
    candidates = df.loc[diffs == min_diff]
    chosen = candidates.sort_values('confidence', ascending=False).iloc[0]
    return pd.Series({'chosen_conf': float(chosen['confidence']), 'chosen_count': int(chosen['count'])})

chosen = (sweep_df.groupby('orthomosaic')
          .apply(lambda d: choose_conf_for_counts(d.sort_values('confidence'), target_count))
          .reset_index())

print("\nChosen thresholds per orthomosaic:")
display(chosen)

# Save chosen crowns
print("\nSaving crowns with chosen thresholds...")
export_rows = []

for _, r in chosen.iterrows():
    stem = r['orthomosaic']
    conf = float(r['chosen_conf'])
    base = bases.get(stem)
    if base is None:
        continue
    
    # Apply chosen threshold
    g = base.copy()
    if fixed_simplify and fixed_simplify > 0:
        g = g.set_geometry(g.geometry.simplify(fixed_simplify))
    g = clean_crowns(g, fixed_iou, conf)
    g = g[g.is_valid]
    
    # Save
    out_file = os.path.join(output_crowns_dir, out_name_for(stem))
    try:
        g.to_file(out_file, driver='GPKG')
        export_rows.append({
            'orthomosaic': stem, 
            'confidence': conf, 
            'count': int(len(g)), 
            'output': out_file
        })
        print(f"  {stem}: conf={conf:.2f}, count={len(g)} → {out_file}")
    except Exception as e:
        print(f"  Failed to save {stem}: {e}")

print(f"\nAll crowns saved to {output_crowns_dir}")

# Summary
summary = pd.DataFrame(export_rows)
display(summary)

# Multi-threshold crown storage format (proposed)
**Goal:** store crowns for multiple confidence thresholds per orthomosaic without overwriting existing single-threshold GPKGs.

**Format:** one GPKG per orthomosaic, with one layer per confidence threshold.

- **File name**: `output/input_crowns_multithreshold/OM{n}_multithreshold.gpkg`
- **Layer naming**: `conf_0p35`, `conf_0p40`, … (decimal with `p` separator)
- **Schema**:
  - Original crown attributes from Detectree2
  - Added attributes:
    - `orthomosaic` (string)
    - `confidence` (float)
    - `threshold_tag` (string, e.g., `conf_0p35`)
- **Sidecar metadata**: `output/input_crowns_multithreshold/OM{n}_multithreshold.meta.json`
  - Fields: `orthomosaic`, `gpkg_path`, `thresholds`, `layer_counts`, `created_at`

**How to use later:**
1. Load the GPKG and list layers. Each layer is one threshold.
2. Choose thresholds based on downstream tracking logic or aggregate across layers.
3. Use the sidecar JSON to quickly discover available thresholds and counts.

This preserves your current single-threshold GPKGs and adds a separate multi-threshold store for feature engineering and tracking experiments.

In [ ]:
# Build multi-threshold crown store for OM1 (test run)
import os
import json
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime
from detectree2.models.outputs import stitch_crowns, clean_crowns

# Settings
om_stem = "sit_om1"
thresholds = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
fixed_iou = 0.7
fixed_simplify = 0.3

multi_dir = os.path.join(project_root, "output", "input_crowns_multithreshold")
os.makedirs(multi_dir, exist_ok=True)
gpkg_path = os.path.join(multi_dir, "OM1_multithreshold.gpkg")
meta_path = os.path.join(multi_dir, "OM1_multithreshold.meta.json")

def _layer_name(conf: float) -> str:
    return f"conf_{str(conf).replace('.', 'p')}"

def _clean_bad_geojson_names(geo_folder: str) -> int:
    bad = glob.glob(os.path.join(geo_folder, '*_None.geojson'))
    for p in bad:
        try:
            os.remove(p)
        except Exception as e:
            print(f"Warning: could not remove {p}: {e}")
    return len(bad)

geo_folder = os.path.join(tiles_dir, om_stem, "predictions_geo")
if not (os.path.isdir(geo_folder) and glob.glob(os.path.join(geo_folder, '*.geojson'))):
    print(f"predictions_geo missing for {om_stem}; running detection...")
    om_path = os.path.join(om_dir, f"{om_stem}.tif")
    if not os.path.exists(om_path):
        raise FileNotFoundError(f"Missing orthomosaic: {om_path}")
    _ = tree_crown_detection_pipeline(
        img_path=om_path,
        tiles_path=os.path.join(tiles_dir, om_stem),
        trained_model_path=model_path,
        buffer=20,
        tile_width=45,
        tile_height=45,
        dtype_bool=True,
        device="cpu",
    )

if not (os.path.isdir(geo_folder) and glob.glob(os.path.join(geo_folder, '*.geojson'))):
    raise FileNotFoundError(f"Missing predictions_geo for {om_stem}: {geo_folder}")

removed = _clean_bad_geojson_names(geo_folder)
if removed:
    print(f"Removed {removed} invalid geojson files in {geo_folder}")

# Stitch once, then filter per threshold
base = stitch_crowns(geo_folder, 1)
base = base[base.is_valid]

layer_counts = {}
for conf in thresholds:
    g = base.copy()
    if fixed_simplify and fixed_simplify > 0:
        g = g.set_geometry(g.geometry.simplify(fixed_simplify))
    g = clean_crowns(g, fixed_iou, conf)
    g = g[g.is_valid]
    g["orthomosaic"] = om_stem
    g["confidence"] = float(conf)
    layer = _layer_name(conf)
    g["threshold_tag"] = layer
    g.to_file(gpkg_path, layer=layer, driver="GPKG")
    layer_counts[layer] = int(len(g))
    print(f"Saved layer {layer}: {len(g)} crowns")

meta = {
    "orthomosaic": om_stem,
    "gpkg_path": gpkg_path,
    "thresholds": thresholds,
    "layer_counts": layer_counts,
    "created_at": datetime.utcnow().isoformat() + "Z",
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"\nWrote: {gpkg_path}")
print(f"Wrote: {meta_path}")

In [ ]:
# Verification: list layers and show counts
import fiona
layers = fiona.listlayers(gpkg_path)
print("Layers:", layers)

counts = []
for layer in layers:
    g = gpd.read_file(gpkg_path, layer=layer)
    counts.append({"layer": layer, "count": int(len(g)), "conf": float(g["confidence"].iloc[0])})

counts_df = pd.DataFrame(counts).sort_values("conf")
display(counts_df)

In [ ]:
# Visualization: counts vs confidence + sample map
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.plot(counts_df["conf"], counts_df["count"], marker="o")
plt.xlabel("Confidence threshold")
plt.ylabel("Crown count")
plt.title("OM1 crowns by threshold")
plt.grid(True, alpha=0.3)
plt.show()

# Map a sample layer (highest confidence)
top_layer = counts_df.sort_values("conf", ascending=False).iloc[0]["layer"]
g_top = gpd.read_file(gpkg_path, layer=top_layer)
ax = g_top.plot(figsize=(5, 5), color="#2ca02c", alpha=0.6, edgecolor="black", linewidth=0.3)
ax.set_title(f"OM1 crowns: {top_layer}")
ax.set_axis_off()

In [ ]:
# Batch: build multi-threshold crown stores for all orthomosaics
import os
import json
import glob
from datetime import datetime

thresholds = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
fixed_iou = 0.7
fixed_simplify = 0.3

multi_dir = os.path.join(project_root, "output", "input_crowns_multithreshold")
os.makedirs(multi_dir, exist_ok=True)

def _layer_name(conf: float) -> str:
    return f"conf_{str(conf).replace('.', 'p')}"

def _clean_bad_geojson_names(geo_folder: str) -> int:
    bad = glob.glob(os.path.join(geo_folder, '*_None.geojson'))
    for p in bad:
        try:
            os.remove(p)
        except Exception as e:
            print(f"Warning: could not remove {p}: {e}")
    return len(bad)

def _om_to_tag(stem: str) -> str:
    m = om_num_pat.search(stem)
    if m:
        return f"OM{int(m.group(1))}"
    return stem

for om_path in om_paths:
    stem = os.path.splitext(os.path.basename(om_path))[0]
    tag = _om_to_tag(stem)
    gpkg_path = os.path.join(multi_dir, f"{tag}_multithreshold.gpkg")
    meta_path = os.path.join(multi_dir, f"{tag}_multithreshold.meta.json")
    geo_folder = os.path.join(tiles_dir, stem, "predictions_geo")

    if not (os.path.isdir(geo_folder) and glob.glob(os.path.join(geo_folder, '*.geojson'))):
        print(f"predictions_geo missing for {stem}; running detection...")
        _ = tree_crown_detection_pipeline(
            img_path=om_path,
            tiles_path=os.path.join(tiles_dir, stem),
            trained_model_path=model_path,
            buffer=20,
            tile_width=45,
            tile_height=45,
            dtype_bool=True,
            device="cpu",
        )

    if not (os.path.isdir(geo_folder) and glob.glob(os.path.join(geo_folder, '*.geojson'))):
        print(f"Skipping {stem}: predictions not found")
        continue

    removed = _clean_bad_geojson_names(geo_folder)
    if removed:
        print(f"Removed {removed} invalid geojson files in {geo_folder}")

    base = stitch_crowns(geo_folder, 1)
    base = base[base.is_valid]

    layer_counts = {}
    for conf in thresholds:
        g = base.copy()
        if fixed_simplify and fixed_simplify > 0:
            g = g.set_geometry(g.geometry.simplify(fixed_simplify))
        g = clean_crowns(g, fixed_iou, conf)
        g = g[g.is_valid]
        g["orthomosaic"] = stem
        g["confidence"] = float(conf)
        layer = _layer_name(conf)
        g["threshold_tag"] = layer
        g.to_file(gpkg_path, layer=layer, driver="GPKG")
        layer_counts[layer] = int(len(g))
    
    meta = {
        "orthomosaic": stem,
        "gpkg_path": gpkg_path,
        "thresholds": thresholds,
        "layer_counts": layer_counts,
        "created_at": datetime.utcnow().isoformat() + "Z",
    }
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"Saved {tag}: {gpkg_path}")

In [ ]:
# Multi-threshold export (0.15 to 0.65) + verification + OM1 visual checks
import os
import re
import json
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import fiona
import matplotlib.pyplot as plt
from datetime import datetime
from detectree2.models.outputs import stitch_crowns, clean_crowns

# Fix PROJ data directory issue
import pyproj

def _ensure_proj_data_dir():
    try:
        _ = pyproj.datadir.get_data_dir()
        return
    except Exception:
        pass

    candidates = []

    env_proj = os.environ.get('PROJ_DATA') or os.environ.get('PROJ_LIB')
    if env_proj:
        candidates.append(env_proj)

    for p in [
        os.path.expanduser('~/anaconda3/share/proj'),
        os.path.expanduser('~/miniconda3/share/proj'),
        '/opt/homebrew/share/proj',
        '/usr/local/share/proj',
    ]:
        candidates.append(p)

    # Try conda base if available
    try:
        import subprocess
        result = subprocess.run(['conda', 'info', '--base'], capture_output=True, text=True)
        conda_base = result.stdout.strip()
        if conda_base:
            candidates.append(os.path.join(conda_base, 'share', 'proj'))
    except Exception:
        pass

    for cand in candidates:
        if cand and os.path.exists(cand):
            pyproj.datadir.set_data_dir(cand)
            os.environ['PROJ_DATA'] = cand
            os.environ['PROJ_LIB'] = cand
            print(f'Set PROJ data directory: {cand}')
            return

    raise RuntimeError('Valid PROJ data directory not found; set PROJ_DATA/PROJ_LIB manually.')

_ensure_proj_data_dir()

project_root = '/Users/hbot07/VS Code/Drone-Phenology-Monitoring'
input_dir = os.path.join(project_root, 'input')
om_dir = os.path.join(input_dir, 'input_om')
tiles_dir = os.path.join(input_dir, 'tiles')
model_path = os.path.join(input_dir, 'detectree_models', '250312_flexi.pth')

om_paths = sorted(glob.glob(os.path.join(om_dir, '*.tif')))
if not om_paths:
    raise FileNotFoundError(f'No orthomosaics found in {om_dir}')

thresholds = [round(x, 2) for x in np.arange(0.15, 0.651, 0.05)]
fixed_iou = 0.7
fixed_simplify = 0.3

multi_dir = os.path.join(project_root, 'output', 'input_crowns_multithreshold_min0p15')
os.makedirs(multi_dir, exist_ok=True)

om_num_pat = re.compile(r'sit_om(\d+)', re.IGNORECASE)
def _om_to_tag(stem: str) -> str:
    m = om_num_pat.search(stem)
    if m:
        return f'OM{int(m.group(1))}'
    return stem

def _layer_name(conf: float) -> str:
    return f"conf_{str(round(conf, 2)).replace('.', 'p')}"

def _clean_bad_geojson_names(geo_folder: str) -> int:
    bad = glob.glob(os.path.join(geo_folder, '*_None.geojson'))
    for p in bad:
        try:
            os.remove(p)
        except Exception as e:
            print(f'Warning: could not remove {p}: {e}')
    return len(bad)

verification_rows = []

for om_path in om_paths:
    stem = os.path.splitext(os.path.basename(om_path))[0]
    tag = _om_to_tag(stem)
    gpkg_path = os.path.join(multi_dir, f'{tag}_multithreshold.gpkg')
    meta_path = os.path.join(multi_dir, f'{tag}_multithreshold.meta.json')
    geo_folder = os.path.join(tiles_dir, stem, 'predictions_geo')

    if not (os.path.isdir(geo_folder) and glob.glob(os.path.join(geo_folder, '*.geojson'))):
        print(f'predictions_geo missing for {stem}; running detection...')
        _ = tree_crown_detection_pipeline(
            img_path=om_path,
            tiles_path=os.path.join(tiles_dir, stem),
            trained_model_path=model_path,
            buffer=20,
            tile_width=45,
            tile_height=45,
            dtype_bool=True,
            device='cpu',
        )

    if not (os.path.isdir(geo_folder) and glob.glob(os.path.join(geo_folder, '*.geojson'))):
        print(f'Skipping {stem}: predictions not found')
        continue

    removed = _clean_bad_geojson_names(geo_folder)
    if removed:
        print(f'Removed {removed} invalid geojson files in {geo_folder}')

    base = stitch_crowns(geo_folder, 1)
    base = base[base.is_valid]

    if os.path.exists(gpkg_path):
        os.remove(gpkg_path)

    layer_counts = {}
    for conf in thresholds:
        g = base.copy()
        if fixed_simplify and fixed_simplify > 0:
            g = g.set_geometry(g.geometry.simplify(fixed_simplify))
        g = clean_crowns(g, fixed_iou, conf)
        g = g[g.is_valid]
        g['orthomosaic'] = stem
        g['confidence'] = float(conf)
        layer = _layer_name(conf)
        g['threshold_tag'] = layer
        g.to_file(gpkg_path, layer=layer, driver='GPKG')
        layer_counts[layer] = int(len(g))

    meta = {
        'orthomosaic': stem,
        'gpkg_path': gpkg_path,
        'thresholds': thresholds,
        'layer_counts': layer_counts,
        'created_at': datetime.utcnow().isoformat() + 'Z',
    }
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)

    # Verification checks
    layers = fiona.listlayers(gpkg_path)
    expected_layers = [_layer_name(c) for c in thresholds]
    missing = sorted(set(expected_layers) - set(layers))

    sample_checks = []
    for lyr in expected_layers:
        if lyr in layers:
            lg = gpd.read_file(gpkg_path, layer=lyr)
            has_cols = all(col in lg.columns for col in ['orthomosaic', 'confidence', 'threshold_tag'])
            vals_ok = True
            if len(lg) > 0:
                vals_ok = (
                    (lg['orthomosaic'].astype(str) == stem).all()
                    and np.isclose(lg['confidence'].astype(float), float(lyr.replace('conf_', '').replace('p', '.'))).all()
                    and (lg['threshold_tag'].astype(str) == lyr).all()
                )
            sample_checks.append(has_cols and vals_ok)
        else:
            sample_checks.append(False)
    attr_ok = all(sample_checks)

    verification_rows.append({
        'orthomosaic': stem,
        'tag': tag,
        'gpkg': gpkg_path,
        'meta': meta_path,
        'layers_found': len(layers),
        'layers_expected': len(expected_layers),
        'missing_layers': ', '.join(missing) if missing else '',
        'attrs_and_values_ok': bool(attr_ok),
        'total_crowns_all_layers': int(sum(layer_counts.values())),
    })

verify_df = pd.DataFrame(verification_rows).sort_values('tag')
display(verify_df)

if verify_df.empty:
    raise RuntimeError('No outputs were generated. Check predictions and model paths.')

if (verify_df['layers_found'] != verify_df['layers_expected']).any() or (~verify_df['attrs_and_values_ok']).any():
    raise AssertionError('Verification failed for one or more orthomosaics; inspect verify_df.')

print(f"\nGenerated and verified multi-threshold outputs in: {multi_dir}")

# OM1 visualization for thresholds 0.15 to 0.45 step 0.1
om1_gpkg = os.path.join(multi_dir, 'OM1_multithreshold.gpkg')
if not os.path.exists(om1_gpkg):
    raise FileNotFoundError(f'OM1 output not found: {om1_gpkg}')

viz_thresholds = [0.15, 0.25, 0.35, 0.45]
viz_layers = [_layer_name(c) for c in viz_thresholds]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, conf, lyr in zip(axes, viz_thresholds, viz_layers):
    g_lyr = gpd.read_file(om1_gpkg, layer=lyr)
    if len(g_lyr) == 0:
        ax.set_title(f'OM1 - conf {conf:.2f} (0 crowns)')
        ax.axis('off')
        continue
    g_lyr.plot(ax=ax, color='#2ca02c', alpha=0.5, edgecolor='black', linewidth=0.2)
    ax.set_title(f'OM1 - conf {conf:.2f} ({len(g_lyr)} crowns)')
    ax.set_axis_off()

plt.tight_layout()
plt.show()

# Quick tabular counts for the same visualization thresholds
om1_counts = []
for conf, lyr in zip(viz_thresholds, viz_layers):
    g_lyr = gpd.read_file(om1_gpkg, layer=lyr)
    om1_counts.append({'confidence': conf, 'layer': lyr, 'count': int(len(g_lyr))})

display(pd.DataFrame(om1_counts))

In [1]:
# LHC area: multi-threshold crown detection and export
import os
import re
import json
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import fiona
from datetime import datetime

import torch
from detectree2.preprocessing.tiling import tile_data
from detectree2.models.train import setup_cfg
from detectree2.models.predict import predict_on_data
from detectree2.models.outputs import project_to_geojson, stitch_crowns, clean_crowns
from detectron2.engine import DefaultPredictor

# Fix PROJ data directory issue
import pyproj

def _ensure_proj_data_dir():
    try:
        _ = pyproj.datadir.get_data_dir()
        return
    except Exception:
        pass

    candidates = []

    env_proj = os.environ.get('PROJ_DATA') or os.environ.get('PROJ_LIB')
    if env_proj:
        candidates.append(env_proj)

    for p in [
        os.path.expanduser('~/anaconda3/share/proj'),
        os.path.expanduser('~/miniconda3/share/proj'),
        '/opt/homebrew/share/proj',
        '/usr/local/share/proj',
    ]:
        candidates.append(p)

    try:
        import subprocess
        result = subprocess.run(['conda', 'info', '--base'], capture_output=True, text=True)
        conda_base = result.stdout.strip()
        if conda_base:
            candidates.append(os.path.join(conda_base, 'share', 'proj'))
    except Exception:
        pass

    for cand in candidates:
        if cand and os.path.exists(cand):
            pyproj.datadir.set_data_dir(cand)
            os.environ['PROJ_DATA'] = cand
            os.environ['PROJ_LIB'] = cand
            print(f'Set PROJ data directory: {cand}')
            return

    raise RuntimeError('Valid PROJ data directory not found; set PROJ_DATA/PROJ_LIB manually.')


def tile_orthomosaic(img_path, tiles_path, buffer=20, tile_width=45, tile_height=45, dtype_bool=True):
    os.makedirs(tiles_path, exist_ok=True)
    tile_data(img_path, tiles_path, buffer, tile_width, tile_height, dtype_bool=dtype_bool)


def setup_detection_config(trained_model_path, device='cpu'):
    cfg = setup_cfg(update_model=trained_model_path)
    try:
        cfg.MODEL.DEVICE = device
    except Exception:
        pass
    return cfg


def predict_tree_crowns(tiles_path, cfg):
    predictor = DefaultPredictor(cfg)
    predict_on_data(directory=tiles_path, predictor=predictor)


def reproject_predictions(tiles_path):
    predictions_folder = os.path.join(tiles_path, 'predictions')
    geo_predictions_folder = os.path.join(tiles_path, 'predictions_geo')
    project_to_geojson(tiles_path, predictions_folder, geo_predictions_folder)
    return geo_predictions_folder


def tree_crown_detection_pipeline(img_path, tiles_path, trained_model_path,
                                  buffer=20, tile_width=45, tile_height=45, dtype_bool=True,
                                  device='cpu'):
    tile_orthomosaic(img_path, tiles_path, buffer, tile_width, tile_height, dtype_bool)
    cfg = setup_detection_config(trained_model_path, device=device)
    predict_tree_crowns(tiles_path, cfg)
    geo_predictions_folder = reproject_predictions(tiles_path)
    return geo_predictions_folder


def _layer_name(conf: float) -> str:
    return f"conf_{str(round(conf, 2)).replace('.', 'p')}"


def _clean_bad_geojson_names(geo_folder: str) -> int:
    bad = glob.glob(os.path.join(geo_folder, '*_None.geojson'))
    for p in bad:
        try:
            os.remove(p)
        except Exception as e:
            print(f'Warning: could not remove {p}: {e}')
    return len(bad)


def _om_to_tag(stem: str, pattern: re.Pattern) -> str:
    m = pattern.search(stem)
    if m:
        return f'OM{int(m.group(1))}'
    return stem


# Thread controls (CPU inference)
os.environ['OMP_NUM_THREADS'] = '6'
os.environ['MKL_NUM_THREADS'] = '6'
os.environ['OPENBLAS_NUM_THREADS'] = '6'
os.environ['NUMEXPR_NUM_THREADS'] = '6'
torch.set_num_threads(6)
torch.set_num_interop_threads(6)

_ensure_proj_data_dir()

project_root = '/Users/hbot07/VS Code/Drone-Phenology-Monitoring'
input_dir = os.path.join(project_root, 'input')
om_dir = os.path.join(input_dir, 'input_om_lhc')
tiles_dir = os.path.join(input_dir, 'tiles_lhc')
model_path = os.path.join(input_dir, 'detectree_models', '250312_flexi.pth')

om_paths = sorted(glob.glob(os.path.join(om_dir, '*.tif')))
if not om_paths:
    raise FileNotFoundError(f'No orthomosaics found in {om_dir}')

# Validate the newly added LHC orthomosaics are included in this run.
required_new_oms = [
    'odm_orthophoto7_03_26.tif',
    'odm_orthophoto9_11_25.tif',
    'odm_orthophoto20_11_25.tif',
    'odm_orthophoto25_10_25.tif',
    'odm_orthophoto26_11_25.tif',
]
present_om_names = {os.path.basename(p) for p in om_paths}
missing_new = sorted(set(required_new_oms) - present_om_names)
if missing_new:
    raise FileNotFoundError(f'Required new OMs not found in {om_dir}: {missing_new}')

print(f'Total LHC orthomosaics to process: {len(om_paths)}')
print('Newly added OMs confirmed:')
for name in required_new_oms:
    print(f'  - {name}')

thresholds = [round(x, 2) for x in np.arange(0.15, 0.651, 0.05)]
fixed_iou = 0.7
fixed_simplify = 0.3

multi_dir = os.path.join(project_root, 'output', 'input_crowns_multithreshold_lhc_10Mar26')
os.makedirs(multi_dir, exist_ok=True)

om_num_pat = re.compile(r'sit_om(\d+)', re.IGNORECASE)
verification_rows = []

for om_path in om_paths:
    stem = os.path.splitext(os.path.basename(om_path))[0]
    tag = _om_to_tag(stem, om_num_pat)
    gpkg_path = os.path.join(multi_dir, f'{tag}_multithreshold.gpkg')
    meta_path = os.path.join(multi_dir, f'{tag}_multithreshold.meta.json')
    geo_folder = os.path.join(tiles_dir, stem, 'predictions_geo')

    if not (os.path.isdir(geo_folder) and glob.glob(os.path.join(geo_folder, '*.geojson'))):
        print(f'predictions_geo missing for {stem}; running detection...')
        _ = tree_crown_detection_pipeline(
            img_path=om_path,
            tiles_path=os.path.join(tiles_dir, stem),
            trained_model_path=model_path,
            buffer=20,
            tile_width=45,
            tile_height=45,
            dtype_bool=True,
            device='cpu',
        )

    if not (os.path.isdir(geo_folder) and glob.glob(os.path.join(geo_folder, '*.geojson'))):
        print(f'Skipping {stem}: predictions not found')
        continue

    removed = _clean_bad_geojson_names(geo_folder)
    if removed:
        print(f'Removed {removed} invalid geojson files in {geo_folder}')

    base = stitch_crowns(geo_folder, 1)
    base = base[base.is_valid]

    if os.path.exists(gpkg_path):
        os.remove(gpkg_path)

    layer_counts = {}
    for conf in thresholds:
        g = base.copy()
        if fixed_simplify and fixed_simplify > 0:
            g = g.set_geometry(g.geometry.simplify(fixed_simplify))
        g = clean_crowns(g, fixed_iou, conf)
        g = g[g.is_valid]
        g['orthomosaic'] = stem
        g['confidence'] = float(conf)
        layer = _layer_name(conf)
        g['threshold_tag'] = layer
        g.to_file(gpkg_path, layer=layer, driver='GPKG')
        layer_counts[layer] = int(len(g))

    meta = {
        'orthomosaic': stem,
        'gpkg_path': gpkg_path,
        'thresholds': thresholds,
        'layer_counts': layer_counts,
        'created_at': datetime.utcnow().isoformat() + 'Z',
    }
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)

    layers = fiona.listlayers(gpkg_path)
    expected_layers = [_layer_name(c) for c in thresholds]
    missing = sorted(set(expected_layers) - set(layers))

    verification_rows.append({
        'orthomosaic': stem,
        'tag': tag,
        'gpkg': gpkg_path,
        'meta': meta_path,
        'layers_found': len(layers),
        'layers_expected': len(expected_layers),
        'missing_layers': ', '.join(missing) if missing else '',
        'total_crowns_all_layers': int(sum(layer_counts.values())),
    })

verify_df = pd.DataFrame(verification_rows).sort_values('orthomosaic')
display(verify_df)

if verify_df.empty:
    raise RuntimeError('No outputs were generated. Check predictions and model paths.')

if (verify_df['layers_found'] != verify_df['layers_expected']).any():
    raise AssertionError('Layer verification failed for one or more orthomosaics; inspect verify_df.')

print(f'Generated and verified multi-threshold outputs in: {multi_dir}')

Total LHC orthomosaics to process: 9
Newly added OMs confirmed:
  - odm_orthophoto7_03_26.tif
  - odm_orthophoto9_11_25.tif
  - odm_orthophoto20_11_25.tif
  - odm_orthophoto25_10_25.tif
  - odm_orthophoto26_11_25.tif
clearn_crowns: Performing spatial join...


clean_crowns: Processing candidate pairs: 100%|██████████| 1386/1386 [00:00<00:00, 3879.87it/s]


clearn_crowns: Performing spatial join...


clean_crowns: Processing candidate pairs: 100%|██████████| 678/678 [00:00<00:00, 4336.26it/s]


clearn_crowns: Performing spatial join...


...

clean_crowns: Processing candidate pairs: 100%|██████████| 78/78 [00:00<00:00, 4213.26it/s]


clearn_crowns: Performing spatial join...


clean_crowns: Processing candidate pairs: 100%|██████████| 72/72 [00:00<00:00, 4142.18it/s]


,orthomosaic,tag,gpkg,meta,layers_found,layers_expected,missing_layers,total_crowns_all_layers
0,odm_orthophoto11_1_26,odm_orthophoto11_1_26,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,11,11,,1032
1,odm_orthophoto20_02_26,odm_orthophoto20_02_26,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,11,11,,1027
2,odm_orthophoto20_11_25,odm_orthophoto20_11_25,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,11,11,,1197
3,odm_orthophoto25_10_25,odm_orthophoto25_10_25,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,11,11,,1049
4,odm_orthophoto26_11_25,odm_orthophoto26_11_25,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,11,11,,930
5,odm_orthophoto4_02_26,odm_orthophoto4_02_26,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,11,11,,1091
6,odm_orthophoto7_03_26,odm_orthophoto7_03_26,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,11,11,,888
7,odm_orthophoto9_11_25,odm_orthophoto9_11_25,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,11,11,,929
8,odm_orthophoto_9_12_25,odm_orthophoto_9_12_25,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,/Users/hbot07/VS Code/Drone-Phenology-Monitori...,11,11,,876


Generated and verified multi-threshold outputs in: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/input_crowns_multithreshold_lhc_10Mar26
